# Finally Solutions

Solutions for `exercises.ipynb`. Try the exercises first — peek here only after you've attempted each question.

## Part 1 — Warm-up

**1. Your first `finally`.** Nothing failed, yet `finally` still ran — that is the whole promise.

In [ ]:
try:
    result = 6 * 7
    print(result)
finally:
    print("experiment over")

**2. Cleanup after a caught error.** Handling an error does not cancel cleanup — both blocks speak, in order.

In [ ]:
try:
    value = int("not today")
except ValueError:
    print("sorry - that is not a number")
finally:
    print("bench wiped down")

**3. Full anatomy, happy path.** Nothing fails, so `except` stays silent, `else` enjoys the success and `finally` signs the audit log.

In [ ]:
n = 21
try:
    double = n * 2
except ValueError:
    print("unexpected problem")
else:
    print("double =", double)
finally:
    print("audit logged")

## Part 2 — Practice

**4. Three paths through one function.** Success, math error, wrong type — three exits, and `finally` attends all three.

In [ ]:
def safe_divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print("  except -> cannot divide by zero")
    except TypeError:
        print("  except -> both operands must be numbers")
    else:
        print(f"  else   -> success, result = {result}")
    finally:
        print("  finally-> attempt finished")


safe_divide(12, 3)
safe_divide(12, 0)
safe_divide(12, "3")

**5. Close the file by hand.** Opening before `try` means there is always something to close — and closing flushes the text to disk.

In [ ]:
from pathlib import Path

Path("sample_data").mkdir(exist_ok=True)
log_path = Path("sample_data", "exercise_log.txt")

log = open(log_path, "w", encoding="utf-8")     # opened BEFORE the try
try:
    log.write("step 1 done\n")
    log.write("step 2 done\n")
    1 / 0                                       # disaster mid-job!
except ZeroDivisionError as e:
    print("handled:", e)
finally:
    log.close()                                 # ALWAYS runs
    print("file closed?", log.closed)

print(log_path.read_text(encoding="utf-8"))

**6. Nested cleanup order.** The innermost `finally` fires on the way out, *before* any outer handler sees the error.

In [ ]:
def inner_job():
    try:
        raise ValueError("sensor unplugged")
    finally:
        print("inner finally -> buffer flushed")


try:
    inner_job()
except ValueError as e:
    print("outer except ->", e)
    print("(cleanup already happened by the time we got here)")

**7. The `return` trap.** A `return` inside `finally` silently overwrites the in-flight result — keep `finally` free of `return`, `break` and `continue`.

In [ ]:
# RULE: `finally` is for cleanup actions only. A return (or break/continue)
# inside it silently discards whatever the try block was about to produce.


def grade_v1():
    try:
        return "computed grade: A"
    finally:
        return "grade lost!"          # silently wins - NEVER do this


def grade_v2():
    try:
        return "computed grade: A"
    finally:
        print("  (cleanup ran - no return here)")


print("trap   :", grade_v1())
print("correct:", grade_v2())

## Part 3 — Challenge

**8. Release the connection, whatever happens.** Databases, sockets, GPU handles: same law — acquire, use, release in `finally`, guarded against "never opened".

In [ ]:
class FakeDBConnection:
    """A stand-in for a real database driver connection."""

    def query(self, sql):
        if "DROP" in sql.upper():
            raise ValueError("malformed SQL")
        print("   querying:", sql)
        return [("Omar", 91)]

    def close(self):
        print("   connection released")


conn = None
try:
    conn = FakeDBConnection()
    print(conn.query("SELECT name, score FROM students"))
    print(conn.query("DROP TABLE students"))      # boom
except ValueError as e:
    print("handled:", e)
finally:
    if conn is not None:
        conn.close()                              # guaranteed on every path

**9. The four-clause file reader.** Specific handlers diagnose, `else` delivers success-only results, `finally` guarantees closure — no crash among the three files.

In [ ]:
from pathlib import Path

Path("sample_data").mkdir(exist_ok=True)
good = Path("sample_data", "diary.txt")
bad = Path("sample_data", "corrupted.bin")
ghost = Path("sample_data", "ghost.txt")

good.write_text("Dear diary: today finally made sense.\n", encoding="utf-8")
bad.write_bytes(b"\xff\xfe high bytes \xff are definitely not UTF-8 text")


def robust_read(path):
    """Return a file's text, or a clear diagnosis - never a crash."""
    f = None
    try:
        f = open(path, encoding="utf-8")      # may raise FileNotFoundError
        text = f.read()                       # may raise UnicodeDecodeError
    except FileNotFoundError:
        return "[missing file]"
    except UnicodeDecodeError as e:
        return f"[not valid UTF-8: {e.reason}]"
    else:
        return text.strip()                   # reached only on full success
    finally:
        if f is not None and not f.closed:
            f.close()
            print(f"   ({Path(path).name} closed in finally)")


for p in (good, bad, ghost):
    print(p.name, "->", robust_read(p))